In [9]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent, AgentState
from langgraph.runtime import Runtime


In [16]:
import sys
import os

sys.path.append(os.path.abspath(".."))
import tools

In [17]:
import importlib
import tools
importlib.reload(tools)

print(dir(tools))

['DDGS', 'Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'analyze_data', 'calculate', 'json', 'load_dotenv', 're', 'tempfile', 'tool', 'web_search']


In [19]:
from langchain_core.runnables import Runnable
from langchain_core.messages import AIMessage
from langchain_core.language_models import LanguageModelInput

In [60]:
from typing import Any

# define tool list for both models

tool_list_complex = [tools.analyze_data, tools.calculate, tools.web_search]
tool_list_simple = [tools.analyze_data, tools.web_search]

def select_model(state: AgentState, runtime: Runtime) -> Runnable:
    """Choose between qwen3:4b and llama3.1:8b based on conversation length."""
    
    messages = state['messages']
    print("messages", messages)
    print("state", state)
    message_count = len(messages)
    
    if message_count < 10:
        print(f"Using qwen3:4b for {message_count} messages (efficient)")
        return ChatOllama(model="qwen3:4b", temperature=3).bind_tools(tool_list_simple)
    else:
        print(f"Switching to llama3.1:8b for {message_count} messages (advanced)")
        return ChatOllama(model="llama3.1:8b", temperature=3, num_predict=2000).bind_tools(tool_list_complex)

In [61]:
def agent_node(state: AgentState, runtime: Runtime):
    model = select_model(state, runtime)
    response = model.invoke(state["messages"])
    
    return {
        "messages": state["messages"] + [response]
    }

In [62]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import END

In [63]:
tool_node = ToolNode(tool_list_complex)

In [64]:
def should_call_tool(state: AgentState):
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    
    return END

In [65]:
from langgraph.graph import StateGraph



graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    should_call_tool,
    {
        "tools": "tools",
        END: END
    }
)
graph.add_edge("tools", "agent")
agent = graph.compile()

In [ ]:
# not worked
agent = create_agent(select_model, tools=tool_list_complex)

In [66]:
from langchain_core.messages import HumanMessage

print(" === Testing Short Conversation (Should Use Qwen) ===")

result1 = agent.invoke({
    "messages": [
        HumanMessage(content="Search for AI news")
    ]
})


 === Testing Short Conversation (Should Use Qwen) ===
messages [HumanMessage(content='Search for AI news', additional_kwargs={}, response_metadata={}, id='31247b63-ed22-400e-81b7-6218913bdfa6')]
state {'messages': [HumanMessage(content='Search for AI news', additional_kwargs={}, response_metadata={}, id='31247b63-ed22-400e-81b7-6218913bdfa6')]}
Using qwen3:4b for 1 messages (efficient)
messages [HumanMessage(content='Search for AI news', additional_kwargs={}, response_metadata={}, id='31247b63-ed22-400e-81b7-6218913bdfa6'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-01-09T12:05:08.1450061Z', 'done': True, 'done_reason': 'stop', 'total_duration': 470288246300, 'load_duration': 1910511000, 'prompt_eval_count': 321, 'prompt_eval_duration': 7797361000, 'eval_count': 2986, 'eval_duration': 459853930400, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019ba29e-3c9f-7011-9448-50ee1cd53cf9-0', too

In [67]:
result1

{'messages': [HumanMessage(content='Search for AI news', additional_kwargs={}, response_metadata={}, id='31247b63-ed22-400e-81b7-6218913bdfa6'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-01-09T12:05:08.1450061Z', 'done': True, 'done_reason': 'stop', 'total_duration': 470288246300, 'load_duration': 1910511000, 'prompt_eval_count': 321, 'prompt_eval_duration': 7797361000, 'eval_count': 2986, 'eval_duration': 459853930400, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019ba29e-3c9f-7011-9448-50ee1cd53cf9-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current ai advancements and industry news highlights'}, 'id': 'b0ecbd94-d499-44e2-bbf1-5148ae811415', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 321, 'output_tokens': 2986, 'total_tokens': 3307}),
  ToolMessage(content="Search results for 'current ai advancements and industry news highlights':\n\

In [68]:
last_msg = result1["messages"][-1]
print(f"\nShort conversation result: {last_msg.content}")


Short conversation result: Here are some **current AI advancements and industry news highlights** based on recent sources:

1. **Key Development from academia**: Researchers at MIT and IBM Watson have launched an expressive architecture designed to significantly boost state tracking and sequential reasoning capabilities in Large Language Models (LLMs) when dealing with long texts. [[MIT News](https://news mit.edu topic/artificialintelligence)]

2. **Industry Momentum**: Multiple major tech firms are investing heavily in AI infrastructure, pushing rapid advancements in real-time data analysis, ethical implementation models, and integrated automation systems—fueled by growing market demand since late Q4 2024. [[Reuters, WSJ](https://wwwReuters.com/the-22-techs-and-its-dominant-daily-aiprogram) [More info in tech coverage](https://www.wsj.com/tech/)] 

3. **Global adoption gaps**: Current industry analytics show accelerating but uneven global AI integration—specifically with the Digital 